In [71]:
import os
import sys
import glob
import rasterio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../ProcessEvents/')
from config import CATCHMENT_LOOKUP_DICT, OUT_DIR, CATCHMENTS # MOLLY_DIR_FF, RAINFALL_CSV_DIR, ENSEMBLE_MEMBERS, , CATCHMENTS

In [ ]:
### not sure why this one is missing

In [3]:
all_catchments = set(CATCHMENT_LOOKUP_DICT.keys())

catchments_with_flood_output = []
for catchment_num in all_catchments:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/{catchment_name}.pkl"
    flood_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/Catchment_{catchment_num}/Ens01_{catchment_num}/10cm/flooded_area_5km_total_Ens01_{catchment_num}_10cm.nc"
        
    if os.path.isfile(flood_fp) and os.path.isfile(fp):
        catchments_with_flood_output.append(catchment_num)
    else:
        print(catchment_num)
        print(os.path.isfile(fp))
        print(os.path.isfile(flood_fp))

89
False
True


In [ ]:
### I think saving to csv cuts off some of the data

In [ ]:
######## WITH ADDED SOIL VARS

In [53]:
rainfall_events_all = []
for catchment_num in catchments_with_flood_output:
    if catchment_num not in ['89']:
        catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
        fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/all_events_added_soilvars.csv"
        rainfall_events = pd.read_csv(fp)
        if 'lu_at_peak.2' in rainfall_events.columns:
            print("YES", catchment_num)
    #     if 'start_month' in rainfall_events.columns:
    #         print("YES", catchment_num)
    #         del rainfall_events['start_day']
    #         del rainfall_events['start_hour']
    #         rainfall_events.rename(columns={'start_month': 'month'}, inplace=True)
    #         rainfall_events.rename(columns={'start_year':'start_year'}, inplace=True)
    #         rainfall_events.rename(columns={'start_day':'day'}, inplace=True)
        #rainfall_events_complete = rainfall_events[rainfall_events['mismatch']!=True].copy()
        rainfall_events['catchment_num'] = catchment_num
        if len(rainfall_events) ==0:
            print(catchment_num)
        # print(f"Catchment {catchment_num} has {len(rainfall_events)}, of which {len(rainfall_events) - len(rainfall_events_complete)} are mismatched")
        rainfall_events_all.append(rainfall_events) 
rainfall_events_all_df = pd.concat(rainfall_events_all, ignore_index=True)   
# rainfall_events_all_df = rainfall_events_all_df[rainfall_events_all_df['max_precip']<130].copy()

In [58]:
# dupes = rainfall_events_all_df[rainfall_events_all_df.duplicated(subset=["x_coord", "y_coord"],
#         keep=False)].sort_values(["x_coord", "y_coord"])

# dupes[['month', 'lu_at_peak']][:50]

In [68]:
catchment46 = rainfall_events_all_df[rainfall_events_all_df['catchment_num']=='46'][['catchment_num', 'ens', 'lu_at_peak', 'event_num']]
catchment46[70:100]

,catchment_num,ens,lu_at_peak,event_num
91585,46,1,0.037800,71
91586,46,1,0.037172,72
91587,46,1,0.037433,73
91588,46,1,NaN,74
91589,46,1,0.037170,75
91590,46,1,0.037433,76
91591,46,1,0.038611,77
91592,46,1,0.043586,78
91593,46,1,0.037063,79
91594,46,1,0.037172,80


In [69]:
nulls = rainfall_events_all_df[rainfall_events_all_df['fu_at_peak'].isnull()][['start_year', 'catchment_num', 'ens', 'lu_at_peak', 'event_num']]
nulls

,start_year,catchment_num,ens,lu_at_peak,event_num
91570,2040,46,1,NaN,56
91574,2041,46,1,NaN,60
91588,2049,46,1,NaN,74
91643,2078,46,1,NaN,129
91644,2078,46,1,NaN,130
...,...,...,...,...,...
92908,2078,46,15,NaN,141
92909,2079,46,15,NaN,142
92910,2079,46,15,NaN,143
92911,2080,46,15,NaN,144


In [107]:
rainfall_events_all_df.to_pickle("/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/all_catchments_added_soilvars.pkl")